# 04 — Condição do céu: multitarefa com fine-tuning, capacidade, janela temporal e ensemble

## O que este notebook compra que a RTX 2060 não compra

O alvo primário passou a ser a **condição do céu** (as quatro classes de Escobedo, binadas
em Kt), com a difusa e o k\* treinados ao lado. Até 2026-09-05 nenhum run com fine-tuning
tinha ligado a cabeça de céu: a única medida vinha de embeddings **congelados**
(`f_cen_clsmean_mt`, acurácia balanceada 0,65, macro-F1 0,65). O braço local `ceu`
(ViT-S/14, 224 px, três sementes) mede o que o fine-tuning faz por essa classe; este
notebook mede o que a A100 acrescenta por cima dele, em ordem de esperança fundamentada:

| arm | hipótese | mecanismo | custo (A100-40GB) |
|---|---|---|---|
| **S. portão** | reproduz o `ceu` local | mesma rede, mesma receita; se divergir, nada abaixo vale | ~1 h |
| **B. capacidade** | ViT-B/14 | maior passo da tabela do DINOv2 que sobrevive ao caso LetsMap de fine-tune com poucos rótulos (notebook 02) | ~2 h |
| **T. janela temporal** | 5 frames em 5 min | o que governa a condição do céu é a nuvem se movendo; um frame não mostra movimento. 5× o forward do backbone | ~4 h |
| **R. resolução** | 448 px | grade 32×32, em direção à nativa 37×37 (DINOv2 termina o pré-treino a 518) | ~3 h |
| **E. ensemble** | 5 sementes do vencedor | ataca variância; o ensemble já levou `anneal` de 19,47 a 18,75 W/m² e `sunangle` a MAE 12,45 | ~1 h |

Total ≈ **11–12 h ≈ 60–65 CU** (A100-40GB ≈ 5,4 CU/h; confira em *View resources*).
Cada arm arquiva no Drive **na mesma célula** do treino, porque o timeout por inatividade
só conta quando a execução termina.

## A receita comum (todos os arms)

`CEU_TARGETS` do `_colab_runner`: sky (entropia cruzada) + kindex (k\*, MAE) + dhi (MAE
sobre `DHI / DHI_céu-claro`, Varaschin & Silva 2025 §5.2.6), pesos 1/1/1; `backbone_pooling:
cls+mean` (a difusa é uma integral hemisférica e o CLS descarta sinal de área pequena —
Muthyala et al. 2026); cosseno completo (paciência = teto, Li et al. 2020); `weight_decay`
0,05 (He et al. 2021, Tab. 9); jitter de exposição, ruído e erase (a câmera varia 1340× a
exposição no dia; flip e rotação seguem proibidos, Nie et al. 2021).

## O número que importa

O teste tem **57 % de céu claro** contra 41 % no treino, e o config não tem peso por classe.
Leia primeiro `sky_balanced_accuracy` e `macro_f1`; a acurácia crua premia quem chuta "claro".

## Antes de rodar

1. `BRANCH` abaixo tem de existir no GitHub **com** o `_colab_runner` deste notebook
   (`CEU_TARGETS`, `ensemble_predictions`, `alignment`/`augmentation` no `write_config`).
2. O bundle é o do **`dataset-iso`** com frames (`bundle-iso.tar.gz`, 1,3 GB), exportado por
   `allsky export-colab-bundle -c configs/allsky/data/local_prepare_iso.yaml --include-frames
   --no-include-embeddings`. O `bundle.tar.gz` antigo é do dataset anisotrópico e **não serve**.
3. Preencha `LOCAL_REFERENCE` com o `ensemble/metrics.json` do braço `ceu` local quando ele
   terminar; sem ele o portão do arm S só compara com a referência congelada.

## 1. Runtime e GPU

`Runtime > Change runtime type > A100 GPU`, *High-RAM* desligado: o pico do ViT-B com janela
de 5 frames fica em ~20 GB, longe dos 40.

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout)

## 2. Ambiente

Única célula que não pode vir do `_colab_runner`: é ela que clona o repositório onde ele
mora. O torch CUDA é instalado e **verificado** — o extra `allsky` fixa uma wheel de CPU.

In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/Bruno-Mascarenhas/micrometeorology.git"
BRANCH = "main"  # precisa carregar o _colab_runner com CEU_TARGETS e ensemble_predictions
WORKDIR = "/content/micrometeorology"

if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, WORKDIR], check=True)
subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.14"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "venv", "--python", "3.14", ".venv"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "sync", "--locked", "--extra", "allsky"], cwd=WORKDIR, check=True)
subprocess.run(
    [
        "uv",
        "pip",
        "install",
        "--python",
        ".venv/bin/python",
        "--reinstall",
        "--torch-backend",
        "cu130",
        "torch==2.13.0",
    ],
    cwd=WORKDIR,
    check=True,
)

PY = f"{WORKDIR}/.venv/bin/python"
os.environ["PATH"] = f"{WORKDIR}/.venv/bin:" + os.environ["PATH"]
sys.path.insert(0, f"{WORKDIR}/notebooks/colab")

verify = subprocess.run(
    [PY, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True,
    text=True,
    check=False,
)
print(verify.stdout)
if "True" not in verify.stdout:
    raise RuntimeError("torch sem CUDA — pare e reinstale antes de treinar")

import _colab_runner as runner  # noqa: E402

if not hasattr(runner, "ensemble_predictions") or not hasattr(runner, "CEU_TARGETS"):
    raise RuntimeError(
        f"o _colab_runner de {BRANCH} nao tem CEU_TARGETS/ensemble_predictions — "
        "aponte BRANCH para uma branch que os carregue"
    )

## 3. Dados e artefatos

`stage_bundle` copia o bundle para o SSD local, desempacota e roda `validate-dataset`.
O bundle tem de ser o do **dataset-iso** (disco concêntrico, raio 112 px em 224).

In [ ]:
import os

from google.colab import drive

BUNDLE = "/content/drive/MyDrive/labmim/allsky-mm/bundle-iso.tar.gz"
DATA = "/content/allsky-mm"
ARTIFACTS = "/content/drive/MyDrive/labmim/runs/allsky-ceu"

# Ensemble do braco `ceu` local (output/allsky-mm/experiments/ceu/ensemble/metrics.json):
# copie aqui dhi.rmse e sky.vote.balanced_accuracy quando ele terminar. None = sem portao
# local; o arm S compara so com a referencia congelada.
LOCAL_REFERENCE = {"dhi_rmse": None, "sky_balanced_accuracy": None}
FROZEN_REFERENCE_BALANCED_ACCURACY = 0.65  # f_cen_clsmean_mt, embeddings congelados

drive.mount("/content/drive")
os.makedirs(ARTIFACTS, exist_ok=True)
ROOT = runner.stage_bundle(BUNDLE, DATA, python=PY)

## 4. Hardware e ajustes que dependem dele

`bf16` existe em toda GPU do Colab menos a T4. O probe roda no interpretador do venv.

In [ ]:
import json

probe = subprocess.run(
    [
        PY,
        "-c",
        'import json, sys; sys.path.insert(0, "' + WORKDIR + '/notebooks/colab"); '
        "import _colab_runner as r; print(json.dumps(r.probe_accelerator()))",
    ],
    capture_output=True,
    text=True,
    check=True,
)
HW = json.loads(probe.stdout.strip().splitlines()[-1])
AMP_DTYPE = HW["amp_dtype"]
WORKERS = min(8, HW["cpus"])
print(HW)
print(f"amp={AMP_DTYPE}  workers={WORKERS}")

## 5. A receita comum, e o arm S — o portão

Um `arm()` só, para os cinco arms compartilharem o mesmo caminho e diferirem apenas no que
cada célula sobrescreve. `epochs` = `patience` = 40 para o cosseno anelar de verdade.

**Cada arm é avaliado em dois checkpoints.** Medido no `ceu` local em 2026-09-05: a entropia
cruzada de céu na validação sobe monotonicamente a partir da época 2 (0,61 → 1,03 na época 7)
enquanto o MAE de difusa cai (20,9 → 15,1 W/m²) — a cabeça de classe fica confiante e errada
numa val 2× mais difícil que o teste, e o `val/loss` composto trava o `best.ckpt` cedo. Por isso
`best` (a escolha do monitor) e `last` (o fim do cosseno) entram os dois na tabela, com sufixo
`_last`; qual serve a cada cabeça é um resultado, não uma premissa.

In [ ]:
from pathlib import Path

CFG = Path(WORKDIR) / "configs/allsky/experiments/colab"
OUT = Path("/content/out")
rows = []

MODEL = {
    "backbone_frozen": False,
    "unfreeze_last_n": 12,
    "image_size": 224,
    "backbone_pooling": "cls+mean",
}
TRAIN = {
    "backbone_lr": 1e-5,
    "epochs": 40,
    "batch_size": 64,
    "weight_decay": 0.05,
    "num_workers": WORKERS,
    "amp": {"enabled": True, "dtype": AMP_DTYPE},
    "early_stopping": {"patience": 40, "monitor": "val/loss"},
}
AUGMENTATION = {
    "p_exposure": 0.5,
    "exposure_log2": 0.6,
    "p_noise": 0.3,
    "noise_sigma": 0.01,
    "p_erase": 0.25,
}


def arm(name, seed, note, *, model=None, train=None, alignment=None):
    """Run one arm, archive it in the same cell, and record its metrics row."""
    config = runner.write_config(
        CFG / f"{name}.yaml",
        extends=["../_base.yaml", "../../models/image_only.yaml"],
        name=name,
        output_dir=str(OUT / name),
        seed=seed,
        data_root=ROOT,
        model={**MODEL, **(model or {})},
        train={**TRAIN, **(train or {})},
        targets=runner.CEU_TARGETS,
        alignment=alignment,
        augmentation=AUGMENTATION,
        note=note,
    )
    row = runner.run_experiment(config, python=PY)
    last = runner.run_experiment(config, python=PY, checkpoint="last")
    for key in ("rmse", "mae", "mbe", "sky_balanced_accuracy", "sky_macro_f1"):
        row[f"{key}_last"] = last.get(key)
    print(
        f"{name:<18} {row.get('status')}  bal_acc={row.get('sky_balanced_accuracy')}"
        f" (last {row.get('sky_balanced_accuracy_last')})  rmse={row.get('rmse')}"
        f" (last {row.get('rmse_last')})  mbe={row.get('mbe')}"
    )
    print(" ", runner.archive(str(OUT / name), ARTIFACTS, config=config))
    row["arm"] = name.rsplit("_s", 1)[0]
    rows.append(row)
    return row


for seed in (42, 43, 44):
    arm(f"ceuS_s{seed}", seed, "arm S: o braco ceu local, nesta GPU — o portao de transferencia")

runner.summarise(rows)

## 6. O portão

Compara o arm S com a referência congelada (obrigatório) e com o `ceu` local (quando
preenchido). Sem amostra não há portão: menos de três sementes concluídas pára a execução.

In [ ]:
import numpy as np


def arm_rows(prefix):
    """Concluded rows of one arm."""
    return [r for r in rows if r.get("status") == "ok" and r.get("arm") == prefix]


gate = arm_rows("ceuS")
if len(gate) < 3:
    raise RuntimeError(f"{3 - len(gate)} semente(s) do arm S falharam — nenhum portao medido")
bal = np.array([r["sky_balanced_accuracy"] for r in gate], dtype=float)
rmse = np.array([r["rmse"] for r in gate], dtype=float)
if not (np.isfinite(bal).all() and np.isfinite(rmse).all()):
    raise RuntimeError("metrica nao finita no arm S — diagnostique antes de seguir")
print(f"arm S: acuracia balanceada {bal.mean():.3f} +- {bal.std(ddof=1):.3f}")
print(f"arm S: RMSE de DHI      {rmse.mean():.2f} +- {rmse.std(ddof=1):.2f} W/m2")
if bal.mean() <= FROZEN_REFERENCE_BALANCED_ACCURACY:
    raise RuntimeError(
        f"fine-tuning nao superou a referencia congelada ({FROZEN_REFERENCE_BALANCED_ACCURACY}); "
        "os arms abaixo herdariam o defeito — pare aqui"
    )
local_bal = LOCAL_REFERENCE.get("sky_balanced_accuracy")
local_rmse = LOCAL_REFERENCE.get("dhi_rmse")
if local_bal is not None and abs(bal.mean() - local_bal) > 0.05:
    print(f"ATENCAO: {bal.mean():.3f} vs {local_bal:.3f} local — mais de 5 pontos; investigue")
if local_rmse is not None and abs(rmse.mean() - local_rmse) > 2.0:
    print(f"ATENCAO: {rmse.mean():.2f} vs {local_rmse:.2f} W/m2 local — mais de 2 W/m2; investigue")
print("portao aberto")

## 7. Arm B — capacidade

ViT-B/14, 12 blocos, batch 48 (o notebook 02 mediu ~4 GB). O portão do ViT-L continua o do
02: só vale se B bater S por mais de duas vezes o ruído entre sementes.

In [ ]:
for seed in (42, 43, 44):
    arm(
        f"ceuB_s{seed}",
        seed,
        "arm B: ViT-B/14 sobre a receita do ceu",
        model={"backbone": "dinov2_vitb14"},
        train={"batch_size": 48},
    )

runner.summarise(rows)

## 8. Arm T — janela temporal

Cinco frames dentro de 5 min, no mesmo dia, codificados pelo backbone e reduzidos por média
mascarada (`alignment.strategy: mean_embedding` em modo imagem). É o arm com a hipótese
física mais forte para a **classe de céu** — nuvem quebrada é movimento — e nunca rodou por
custar 5× o forward. Roda sobre o vencedor de S/B.

In [ ]:
def mean_of(prefix, key):
    """Mean of one metric over the concluded seeds of an arm."""
    values = [r[key] for r in arm_rows(prefix) if r.get(key) is not None]
    return float(np.mean(values)) if values else float("nan")


WINNER = (
    "ceuB"
    if mean_of("ceuB", "sky_balanced_accuracy") > mean_of("ceuS", "sky_balanced_accuracy")
    else "ceuS"
)
WINNER_MODEL = {"backbone": "dinov2_vitb14"} if WINNER == "ceuB" else {}
WINNER_BATCH = 48 if WINNER == "ceuB" else 64
print("vencedor S/B pela acuracia balanceada:", WINNER)

for seed in (42, 43, 44):
    arm(
        f"ceuT_s{seed}",
        seed,
        "arm T: janela de 5 frames em 5 min sobre o vencedor de S/B",
        model=WINNER_MODEL,
        train={"batch_size": max(8, WINNER_BATCH // 4)},
        alignment={"strategy": "mean_embedding", "window_minutes": 5.0, "max_frames": 5},
    )

runner.summarise(rows)

## 9. Arm R — resolução

448 px sobre o vencedor de S/B: grade 32×32 em direção à nativa 37×37. Touvron et al. (2019)
avisam que a resolução de teste só rende depois de um fine-tune na resolução alvo — que é
exatamente o que este arm faz. Duas sementes: é o arm mais caro por época.

In [ ]:
for seed in (42, 43):
    arm(
        f"ceuR_s{seed}",
        seed,
        "arm R: 448 px = grade 32x32, em direcao a nativa 37x37",
        model={**WINNER_MODEL, "image_size": 448},
        train={"batch_size": max(4, WINNER_BATCH // 4)},
    )

runner.summarise(rows)

## 10. Arm E — ensemble de cinco sementes do vencedor

Lakshminarayanan et al. usam M = 5. Duas sementes extras do melhor arm até aqui, e então o
`ensemble_predictions` do runner: média das predições para a difusa e o k\*, e para o céu
**dois estimadores** — o voto de maioria da cabeça CE e o Kt reconstruído do k\* médio binado
nos limites de Escobedo. Qual dos dois ganha é um achado. A referência pareada é o arm S.

**O ensemble heterogêneo vem depois, e é o que costuma ganhar.** Medido em 2026-09-05 nos
parquets locais: a correlação de erro entre sementes do mesmo config é ~0,88–0,94, entre
famílias diferentes cai a ~0,68, e `anneal3 + sunangle3` deu 17,46 W/m² contra 18,75 do melhor
ensemble de um braço só. Aqui a diversidade vem da arquitetura (S/B/T/R): a segunda chamada
junta **todos** os membros concluídos.

In [ ]:
BEST = max(("ceuS", "ceuB", "ceuT", "ceuR"), key=lambda p: mean_of(p, "sky_balanced_accuracy"))
print("melhor arm pela acuracia balanceada:", BEST)
ARM_SETTINGS = {
    "ceuS": {"model": {}, "train": {}, "alignment": None},
    "ceuB": {
        "model": {"backbone": "dinov2_vitb14"},
        "train": {"batch_size": 48},
        "alignment": None,
    },
    "ceuT": {
        "model": WINNER_MODEL,
        "train": {"batch_size": max(8, WINNER_BATCH // 4)},
        "alignment": {"strategy": "mean_embedding", "window_minutes": 5.0, "max_frames": 5},
    },
    "ceuR": {
        "model": {**WINNER_MODEL, "image_size": 448},
        "train": {"batch_size": max(4, WINNER_BATCH // 4)},
        "alignment": None,
    },
}

for seed in (45, 46):
    arm(f"{BEST}_s{seed}", seed, f"arm E: membro extra do ensemble de {BEST}", **ARM_SETTINGS[BEST])

members = sorted(Path(ARTIFACTS).glob(f"{BEST}_s*/eval-test/predictions.parquet"))
reference = sorted(Path(ARTIFACTS).glob("ceuS_s*/eval-test/predictions.parquet"))
if len(members) < 2:
    raise RuntimeError(f"{len(members)} membro(s) de {BEST} com predicoes — sem ensemble")
report = runner.ensemble_predictions(
    members, Path(ARTIFACTS) / f"ensemble_{BEST}", reference=reference
)
members_last = sorted(Path(ARTIFACTS).glob(f"{BEST}_s*/eval-test-last/predictions.parquet"))
if len(members_last) >= 2:
    report_last = runner.ensemble_predictions(
        members_last, Path(ARTIFACTS) / f"ensemble_{BEST}_last"
    )
    print(
        f"ensemble dos last.ckpt: RMSE {report_last['dhi']['rmse']:.2f}  "
        f"bal_acc voto {report_last.get('sky', {}).get('vote', {}).get('balanced_accuracy', float('nan')):.3f}"
    )
for estimator, metrics in report.get("sky", {}).items():
    print(
        f"ceu/{estimator:7s} acuracia balanceada {metrics['balanced_accuracy']:.3f}  "
        f"macro-F1 {metrics['macro_f1']:.3f}  MAE ordinal {metrics['ordinal_mae']:.3f}"
    )
dhi = report["dhi"]
print(
    f"difusa: RMSE {dhi['rmse']:.2f}  MAE {dhi['mae']:.2f}  MBE {dhi['mbe']:+.2f} W/m2  (n={len(members)})"
)
if "reference" in report:
    print(f"vs arm S pareado: delta RMSE {report['reference']['rmse_delta']:+.2f} W/m2")

everyone = sorted(Path(ARTIFACTS).glob("ceu[SBTR]_s*/eval-test/predictions.parquet"))
if len(everyone) > len(members):
    hetero = runner.ensemble_predictions(
        everyone, Path(ARTIFACTS) / "ensemble_heterogeneo", reference=reference
    )
    for estimator, metrics in hetero.get("sky", {}).items():
        print(
            f"heterogeneo/{estimator:7s} acuracia balanceada {metrics['balanced_accuracy']:.3f}  "
            f"macro-F1 {metrics['macro_f1']:.3f}"
        )
    print(
        f"heterogeneo ({len(everyone)} membros): RMSE {hetero['dhi']['rmse']:.2f}  "
        f"MAE {hetero['dhi']['mae']:.2f}  MBE {hetero['dhi']['mbe']:+.2f} W/m2"
    )

## 11. Fechamento

Grava o índice da campanha. `sky_balanced_accuracy` vem primeiro na ordenação por ser o
alvo; a difusa fica ao lado para dizer quanto ela pagou pela classe.

In [ ]:
frame = runner.summarise(rows)
if "sky_balanced_accuracy" in frame.columns:
    frame = frame.sort_values("sky_balanced_accuracy", ascending=False, na_position="last")
frame.to_csv(f"{ARTIFACTS}/campanha_ceu.csv", index=False)
summary = {
    "hardware": HW,
    "n_runs": len(rows),
    "melhor_arm": BEST,
    "ensemble": report,
    "heterogeneo": hetero if len(everyone) > len(members) else None,
}
with open(f"{ARTIFACTS}/campanha_ceu_resumo.json", "w") as handle:
    json.dump(summary, handle, indent=2, default=str)
print(frame.to_string())
print("artefatos em", ARTIFACTS)